# Group 8 — International Finance for Clean Energy
AS5004 Foundational Statistics for AI · MCA Semester 1 · Unit 2

This notebook implements the supplied handout: SDG 7.a descriptive/outlier analysis and a country-held-out renewable-share classifier.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay

ROOT = Path.cwd()
if not (ROOT / 'data' / 'Group_08_Dataset.xlsx').exists(): ROOT = ROOT.parent
DATA = ROOT / 'data' / 'Group_08_Dataset.xlsx'
OUT = ROOT / 'outputs'; FIG = OUT / 'figures'; FIG.mkdir(parents=True, exist_ok=True)
df = pd.read_excel(DATA, sheet_name='Data', header=11)
numeric = ['year','clean_energy_finance_all_usd_m','finance_solar_usd_m','finance_wind_usd_m','finance_hydropower_usd_m','finance_bioenergy_usd_m','elec_access_total_pct','renewable_share_tfec_pct','re_capacity_all_w_per_capita']
for c in numeric: df[c] = pd.to_numeric(df[c], errors='coerce')
print(df.shape, 'rows and columns')
print(df.country.nunique(), 'countries;', df.year.min(), 'to', df.year.max())
display(df.head())

## Phase 1 — Data quality audit
The workbook specifies row 12 as the header. Blank cells are retained as missing, not recoded to zero. The raw workbook remains unchanged.

In [ ]:
audit = pd.DataFrame({'dtype':df.dtypes.astype(str),'missing_count':df.isna().sum(),'missing_percent':(df.isna().mean()*100).round(2),'unique_values':df.nunique()})
display(audit)
print('Exact duplicate rows:', df.duplicated().sum())
print('Duplicate country-year keys:', df.duplicated(['iso3','year']).sum())
OUT.mkdir(exist_ok=True); audit.to_csv(OUT/'data_quality_audit.csv')

## Phase 2 — Descriptive statistics
Compute mean, median, mode, variance, standard deviation, IQR, skewness and kurtosis. Compare by continent.

In [ ]:
key = ['clean_energy_finance_all_usd_m','elec_access_total_pct','renewable_share_tfec_pct','re_capacity_all_w_per_capita','finance_solar_usd_m','finance_wind_usd_m','finance_hydropower_usd_m','finance_bioenergy_usd_m']
def stats(s):
    s=s.dropna(); q1,q3=s.quantile([.25,.75]); modes=s.mode()
    return pd.Series({'n':len(s),'mean':s.mean(),'median':s.median(),'mode':', '.join(map(str,modes.head(3).tolist())),'variance':s.var(),'sd':s.std(),'Q1':q1,'Q3':q3,'IQR':q3-q1,'skewness':s.skew(),'kurtosis':s.kurt()})
summary=pd.DataFrame({c:stats(df[c]) for c in key}).T
display(summary); summary.to_csv(OUT/'descriptive_statistics.csv')
grouped=df.groupby('continent')[key[:4]].agg(['count','mean','median'])
display(grouped); grouped.to_csv(OUT/'continent_summary.csv')

## Phase 3 — Zero values and outliers
Z-score flags use |z| > 3. IQR flags use 1.5 × IQR fences. Outliers are not automatically deleted; large financial flows may be substantively important.

In [ ]:
finance_cols=['clean_energy_finance_all_usd_m','finance_solar_usd_m','finance_wind_usd_m','finance_hydropower_usd_m','finance_bioenergy_usd_m']
zero_skew=pd.DataFrame({'n':df[finance_cols].count(),'zeros':(df[finance_cols]==0).sum(),'zero_percent':(df[finance_cols].eq(0).sum()/df[finance_cols].count()*100),'skewness':df[finance_cols].skew()})
display(zero_skew); zero_skew.to_csv(OUT/'finance_zero_skewness.csv')
def outlier_report(col):
    s=df[col].dropna(); z=(s-s.mean())/s.std(); q1,q3=s.quantile([.25,.75]); iqr=q3-q1; zf=z.abs()>3; qf=(s<q1-1.5*iqr)|(s>q3+1.5*iqr)
    return {'n':len(s),'zscore_count':int(zf.sum()),'zscore_percent':float(zf.mean()*100),'iqr_low':float(q1-1.5*iqr),'iqr_high':float(q3+1.5*iqr),'iqr_count':int(qf.sum()),'iqr_percent':float(qf.mean()*100),'both':int((zf&qf).sum())}
outliers=pd.DataFrame({c:outlier_report(c) for c in ['clean_energy_finance_all_usd_m','re_capacity_all_w_per_capita']}).T
display(outliers); outliers.to_csv(OUT/'outlier_results.csv')
top=df.nlargest(15,'clean_energy_finance_all_usd_m')[['country','iso3','continent','year','clean_energy_finance_all_usd_m']]
display(top); top.to_csv(OUT/'top_finance_country_years.csv',index=False)

## Phase 4 — Visualisations and SDG 7.a interpretation
Histograms and boxplots are generated for key variables. Finance/capacity use log1p transforms so zero observations remain valid.

In [ ]:
plot_cols=key
for c in plot_cols:
    s=df[c].dropna(); fig,ax=plt.subplots(1,2,figsize=(10,3.5))
    if 'finance' in c or 'capacity' in c: vals=np.log1p(s.clip(lower=0)); xlabel='log1p(value)'
    else: vals=s; xlabel=c
    ax[0].hist(vals,bins=35,edgecolor='black'); ax[0].set_xlabel(xlabel); ax[0].set_ylabel('Country-years'); ax[0].set_title('Histogram: '+c)
    ax[1].boxplot(vals); ax[1].set_ylabel(xlabel); ax[1].set_title('Boxplot: '+c)
    fig.tight_layout(); fig.savefig(FIG/(c+'_distribution.png'),dpi=150); plt.show()
df['decade']=(df.year//10*10).astype(int).astype(str)+'s'
gd=df.groupby(['continent','decade'])['clean_energy_finance_all_usd_m'].median().reset_index()
fig,ax=plt.subplots(figsize=(9,4))
for name,g in gd.groupby('continent'): ax.plot(g.decade,g.clean_energy_finance_all_usd_m,marker='o',label=name)
ax.set_yscale('symlog',linthresh=1); ax.set_xlabel('Decade'); ax.set_ylabel('Median finance (million constant 2023 US$)'); ax.set_title('Median finance by continent and decade'); ax.legend(); fig.tight_layout(); fig.savefig(FIG/'finance_by_continent_decade.png',dpi=150); plt.show()
country_totals=df.groupby(['country','iso3','continent'])['clean_energy_finance_all_usd_m'].sum().sort_values(ascending=False).reset_index()
display(country_totals.head(15)); country_totals.to_csv(OUT/'country_finance_totals.csv',index=False)

## Phase 5 — Classification model
Target: renewable share above the median. Predictors: log1p finance, electricity access, log1p installed capacity and year. Use complete cases and split by country to prevent leakage. Compare logistic regression against a majority-class baseline.

In [ ]:
target='renewable_share_tfec_pct'; raw_features=['clean_energy_finance_all_usd_m','elec_access_total_pct','re_capacity_all_w_per_capita','year']
m=df[['country','iso3',target]+raw_features].dropna().copy()
m=m[(m.clean_energy_finance_all_usd_m>=0)&(m.re_capacity_all_w_per_capita>=0)]
median_target=m[target].median(); m['label']=(m[target]>median_target).astype(int)
m['log_finance']=np.log1p(m.clean_energy_finance_all_usd_m); m['log_capacity']=np.log1p(m.re_capacity_all_w_per_capita)
features=['log_finance','elec_access_total_pct','log_capacity','year']; X=m[features]; y=m.label; groups=m.iso3
split=GroupShuffleSplit(n_splits=1,test_size=.25,random_state=42); train,test=next(split.split(X,y,groups))
assert set(groups.iloc[train]).isdisjoint(set(groups.iloc[test]))
X_train,X_test=X.iloc[train],X.iloc[test]; y_train,y_test=y.iloc[train],y.iloc[test]
print('Complete rows:',len(m),'Countries:',groups.nunique(),'Median cutoff:',median_target)
print('Train countries:',groups.iloc[train].nunique(),'Test countries:',groups.iloc[test].nunique(),'Overlap:',len(set(groups.iloc[train])&set(groups.iloc[test])))
display(m.label.value_counts(normalize=True).rename('proportion').to_frame())
model=Pipeline([('scale',StandardScaler()),('logistic',LogisticRegression(max_iter=2000,class_weight='balanced',random_state=42))]); model.fit(X_train,y_train); pred=model.predict(X_test)
base=np.full(len(y_test),y_train.mode().iloc[0])
metrics=pd.DataFrame([{'model':'Majority baseline','accuracy':accuracy_score(y_test,base),'precision':precision_score(y_test,base,zero_division=0),'recall':recall_score(y_test,base,zero_division=0)},{'model':'Logistic regression','accuracy':accuracy_score(y_test,pred),'precision':precision_score(y_test,pred,zero_division=0),'recall':recall_score(y_test,pred,zero_division=0)}])
display(metrics); metrics.to_csv(OUT/'model_metrics.csv',index=False)
cm=confusion_matrix(y_test,pred,labels=[0,1]); ConfusionMatrixDisplay(cm,display_labels=['Low','High']).plot(values_format='d'); plt.title('Confusion matrix — held-out countries'); plt.tight_layout(); plt.savefig(FIG/'confusion_matrix.png',dpi=150); plt.show()
evaluation=m.iloc[test][['country','iso3','year',target,'label']].copy(); evaluation['predicted']=pred; evaluation['correct']=evaluation.label==evaluation.predicted; evaluation.to_csv(OUT/'heldout_predictions.csv',index=False)

## Phase 6 — Limitations and conclusion
Discuss missing-data reduction, modelled UN estimates, percentage values clustered near 100, the limitations of a single grouped split, and the fact that association is not causation. Report whether the model meaningfully beats the baseline. Cite UN DESA SDG Global Database and the custodian agencies named in the workbook Data Dictionary. Acknowledge AI assistance as required by your handout.